In [ ]:
# ==== THAM SỐ (cell duy nhất cần sửa) — đủ CLI của src.report, src.stats_test, src.visualize ====
REPO_URL = "https://github.com/NguyenMinhTri24072005/MV_CAAFNet.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/MV_CAAFNet"
DRIVE_DATA_SUBDIR = "data/processed"
LOCAL_DATA_DIR = "/content/processed"  # --data-dir
EXP_ROOT = f"{DRIVE_ROOT}/experiments" # --exp-root
OUT_REPORTS_DIR = f"{DRIVE_ROOT}/reports" # --out-dir (tự mirror về D:\GG_1)

REPORT_CKPT = "last"                   # --ckpt
REPORT_SEED = 42                       # --seed
INCLUDE_SENSITIVITY = False            # --include-sensitivity

STATS_PAIRS = []                       # list (run_a_rel, run_b_rel) -> --a, --b
STATS_METRIC = "dice"                  # --metric
STATS_OUT_DIR = f"{OUT_REPORTS_DIR}/stats"  # --out (stats_test: <STATS_OUT_DIR>/<a>__vs__<b>_<metric>.json)

VIZ_KINDS = ["loss", "dotplot", "diffmap", "gradcam", "weights"] # --kind
VIZ_RUN_DIRS = [                       # --run-dirs
    "luna16/caaf_sobel_s42",
    "luna16/caaf_laplacian_s42",
    "luna16/caaf_hausdorff_s42",
]
VIZ_CKPT = "last"                      # --ckpt
VIZ_SPLIT = "test"                     # --split
VIZ_DATASET_OVERRIDE = None            # --dataset
PATCH_IDS = []                         # --patch-ids (chọn sau cell lọc ứng viên C1–C5 từ patches.csv)
GRADCAM_LAYERS = ["aaf1", "aaf2", "aaf3"] # --layer

# 12 — Thống kê, bảng so sánh & hình (Phase 9)

**Runtime:** Colab Pro **GPU** (chỉ `gradcam` cần GPU/torch). Chỉ sửa **cell tham số** (đủ CLI của `src.report`,
`src.stats_test`, `src.visualize`). Bảng `.md` và hình ghi vào `DRIVE_ROOT/reports/` (tự mirror về máy local).

In [ ]:
# Setup chung: mount Drive → clone/pull repo (GH_TOKEN qua GIT_ASKPASS, không in token) → cài thư viện → in GPU
import json, os, platform, shlex, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/MV_CAAFNet"


def _git_env():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")          # Colab Secrets → không ghi token vào notebook
    except Exception:
        token = None                               # repo public hoặc chưa cấu hình secret
    if token:
        askpass = Path("/content/.git_askpass.sh")
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n  *) printf "%s" "$GH_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GH_TOKEN=token)
    return env


def _git(*args):
    subprocess.run(["git", *args], check=True, env=_git_env())


if Path(REPO_DIR, ".git").is_dir():
    _git("-C", REPO_DIR, "fetch", "--quiet", "origin", BRANCH)
    _git("-C", REPO_DIR, "checkout", "--quiet", BRANCH)
    _git("-C", REPO_DIR, "pull", "--quiet", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)


def run(args):
    """Chạy `python <args>` từ thư mục repo (danh sách tham số → an toàn với đường dẫn có dấu cách)."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


import torch

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"commit {commit} | Python {platform.python_version()} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1024 ** 3:.1f} GB")
else:
    print("⚠ Không có GPU — Runtime → Change runtime type → GPU")

## 1. Bảng so sánh với bài báo (Table 5, 6, 7, seed variance, sensitivity)

In [ ]:
from IPython.display import Image, Markdown, display

args = ["-m", "src.report", "--exp-root", EXP_ROOT, "--out-dir", OUT_REPORTS_DIR, "--ckpt", REPORT_CKPT,
        "--seed", REPORT_SEED]
if INCLUDE_SENSITIVITY:
    args.append("--include-sensitivity")
run(args)
for name in ("compare_table5.md", "compare_table6.md", "compare_table7.md", "seed_variance.md", "sensitivity.md"):
    f = Path(OUT_REPORTS_DIR) / name
    if f.exists():
        display(Markdown(f.read_text(encoding="utf-8")))

## 2. Paired t-test cho các cặp tự chọn (`d_i = a_i − b_i`, đơn vị %)

In [ ]:
for a_rel, b_rel in STATS_PAIRS:
    a = Path(EXP_ROOT) / a_rel / f"eval_test_{REPORT_CKPT}" / "per_patch_metrics.csv"
    b = Path(EXP_ROOT) / b_rel / f"eval_test_{REPORT_CKPT}" / "per_patch_metrics.csv"
    out = Path(STATS_OUT_DIR) / f"{a_rel.replace('/', '_')}__vs__{b_rel.replace('/', '_')}_{STATS_METRIC}.json"
    run(["-m", "src.stats_test", "--a", a, "--b", b, "--metric", STATS_METRIC, "--out", out])
    print(json.dumps(json.loads(out.read_text(encoding="utf-8")), ensure_ascii=False, indent=1))

## 3. Dữ liệu `.h5` cho `diffmap` / `gradcam`

In [ ]:
# Chép .h5 từ Drive về SSD /content (một lần mỗi phiên) và kiểm SHA-256 theo manifest.json
from src.data.pack_h5 import verify_manifest


def stage_dataset(ds, verify=True):
    src = Path(DRIVE_ROOT) / DRIVE_DATA_SUBDIR / ds
    dst = Path(LOCAL_DATA_DIR) / ds
    if not (src / "manifest.json").exists():
        raise FileNotFoundError(f"Chưa có {src}/manifest.json — chạy 01_colab_preprocess.ipynb trước")
    fresh = not ((dst / "manifest.json").exists()
                 and (dst / "manifest.json").read_bytes() == (src / "manifest.json").read_bytes())
    if fresh:
        shutil.rmtree(dst, ignore_errors=True)
        dst.mkdir(parents=True)
        for name in json.loads((src / "manifest.json").read_text(encoding="utf-8"))["files"]:
            shutil.copy2(src / name, dst / name)
        shutil.copy2(src / "manifest.json", dst / "manifest.json")  # chép cuối cùng
    if verify:
        bad = verify_manifest(dst)
        if bad:
            raise RuntimeError(f"SHA-256 sai ở {dst}: {bad} — xóa {dst} rồi chạy lại cell")
    print(f"{ds}: {'đã chép' if fresh else 'đã có sẵn'} → {dst}" + (" | SHA-256 OK" if verify else ""))

from src.utils import load_config

viz_runs = [Path(EXP_ROOT) / r for r in VIZ_RUN_DIRS]
viz_datasets = sorted({VIZ_DATASET_OVERRIDE or load_config(r / "config_resolved.yaml")["dataset"] for r in viz_runs})
for ds in viz_datasets:
    stage_dataset(ds)

## 4. Lọc bán tự động ứng viên tương tự C1–C5 (Table 11–13) từ `patches.csv`

Bộ dữ liệu MEF-PN35 của bài báo không công bố, nên chọn ca **tương tự** trong tập đánh giá theo heuristic hình thái
(§5.1 `compute_patch_morphology`). Xem bảng, chỉnh `PATCH_IDS` ở cell tham số nếu muốn chọn tay; để trống thì dùng gợi ý.

In [ ]:
import pandas as pd

ds = viz_datasets[0]
proc = Path(LOCAL_DATA_DIR) / ds
patches = pd.read_csv(proc / "patches.csv", dtype={"patch_id": str, "patient_id": str})
split_ids = set(json.loads((proc / f"{ds}.json").read_text(encoding="utf-8"))[VIZ_SPLIT])
cand = patches[patches["patch_id"].isin(split_ids)].copy()
cand["touches_border"] = cand["touches_border"].astype(bool)
rules = {
    "C1 bờ rõ": (cand["solidity"] >= 0.95) & ~cand["touches_border"] & cand["diameter_mm"].between(8, 20)
                & (cand["n_other_nodules"] == 0),
    "C2 nốt nhỏ": cand["diameter_mm"] < 6,
    "C3a dính mạch máu": (cand["solidity"] < 0.85) & cand["diameter_mm"].between(6, 15) & ~cand["touches_border"],
    "C3b dính thành ngực": cand["touches_border"] & (cand["mean_hu"] > -300),
    "C4 có hang": (cand["min_hu"] < -600) & (cand["diameter_mm"] >= 10),
    "C5 tua gai": (cand["solidity"] < 0.75) & (cand["diameter_mm"] >= 8),
}
cols = ["patch_id", "diameter_mm", "area_nodule_px", "solidity", "mean_hu", "min_hu", "touches_border", "n_other_nodules"]
suggested = []
for name, mask in rules.items():
    top = cand[mask].sort_values("area_nodule_px", ascending=False).head(3)
    print(f"{name}: {int(mask.sum())} ứng viên")
    display(top[cols])
    if len(top) and top.iloc[0]["patch_id"] not in suggested:
        suggested.append(top.iloc[0]["patch_id"])
PATCH_IDS_USE = PATCH_IDS or suggested
print("PATCH_IDS dùng để vẽ:", PATCH_IDS_USE)

## 5. Hình: Figure 2 (`loss`), Figure 3 (`dotplot`), `diffmap`, `gradcam`, `weights`

In [ ]:
fig_dir = f"{OUT_REPORTS_DIR}/figures"
for kind in VIZ_KINDS:
    for layer in (GRADCAM_LAYERS if kind == "gradcam" else [None]):
        args = ["-m", "src.visualize", "--run-dirs", *[f"{EXP_ROOT}/{r}" for r in VIZ_RUN_DIRS], "--kind", kind,
                "--out-dir", fig_dir, "--data-dir", LOCAL_DATA_DIR, "--ckpt", VIZ_CKPT, "--split", VIZ_SPLIT]
        if kind in ("diffmap", "gradcam") and PATCH_IDS_USE:
            args += ["--patch-ids", *PATCH_IDS_USE]
        if layer:
            args += ["--layer", layer]
        if VIZ_DATASET_OVERRIDE:
            args += ["--dataset", VIZ_DATASET_OVERRIDE]
        run(args)
for png in sorted(Path(fig_dir).glob("*.png")):
    print(png.name)
    display(Image(filename=str(png), width=720))